# Phase 1: Spurgeon Continued Pretraining — Step 7: Training (Notebook B)

This notebook executes PEFT QLoRA continued pretraining on `unsloth/Qwen2.5-3B` using the Hugging Face Dataset compiled in Notebook A. It handles 4-bit quantization, sequence packing (for fast compute), and cross-session checkpoint resumption logic.

## 1. Install Dependencies

In [ ]:
# Install Unsloth and specific patched versions for Kaggle environment
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"

## 2. Model & PEFT Setup

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048
LORA_RANK = 32

# Load base model in 4-bit quantization
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name   = "unsloth/Qwen2.5-3B",
    max_seq_length = MAX_SEQ_LENGTH,
    dtype        = None,
    load_in_4bit = True,
)

# Apply the PEFT LoRA adapter
model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_RANK,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha   = 64,
    lora_dropout = 0, # Critical: set to 0 to enable fast custom Triton kernels
    bias         = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

## 3. Configure Dataset and Training Arguments

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_from_disk
import os

# Load the dataset generated in Notebook A
dataset = load_from_disk("/kaggle/input/datasets/rafaelvieira1/spurgeon-cpt-dataset/spurgeon_dataset")

output_dir = "/kaggle/working/checkpoints"

# RESUMPTION CONFIGURATION
# To resume: mount the output dataset of your previous run (e.g. Run 1) as an Input Dataset
RUN_NUMBER = 1 # Update to 2, 3, etc. for subsequent runs

# If resuming from a previous run output mounted as an input, specify the path here:
# Example: "/kaggle/input/datasets/rafaelvieira1/spurgeon-training-run-1/checkpoints/checkpoint-1500"
PREV_RUN_CHECKPOINT = None 

# Define total target epochs (must be equal to the current run number)
TOTAL_TARGET_EPOCHS = RUN_NUMBER

training_args = TrainingArguments(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 8,   # Effective batch size = 16 (2 * 8)
    num_train_epochs            = TOTAL_TARGET_EPOCHS,
    warmup_steps                = 100,
    learning_rate               = 2e-4,
    lr_scheduler_type           = "cosine",
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    optim                       = "adamw_8bit",
    weight_decay                = 0.01,
    logging_steps               = 50,
    eval_strategy         = "steps",
    eval_steps                  = 500,
    save_strategy               = "steps",
    save_steps                  = 500,
    save_total_limit            = 3,     # Keep the last 3 checkpoints to manage disk space
    output_dir                  = output_dir,
    seed                        = 42,
)

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset["train"],
    eval_dataset       = dataset["test"],
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LENGTH,
    packing            = True, # Packs multiple short texts into a single context window
    args               = training_args,
)

## 4. Launch Training

In [ ]:
if PREV_RUN_CHECKPOINT:
    if os.path.exists(PREV_RUN_CHECKPOINT):
        print(f"Resuming training from checkpoint: {PREV_RUN_CHECKPOINT}")
        trainer.train(resume_from_checkpoint=PREV_RUN_CHECKPOINT)
    else:
        raise FileNotFoundError(f"Checkpoint not found at: {PREV_RUN_CHECKPOINT}. "
                                "Please check the path or ensure the previous run's dataset is mounted.")
else:
    print("Starting training from scratch...")
    trainer.train()

## 5. Save PEFT Adapter Weights

In [ ]:
output_path = f"/kaggle/working/spurgeon_lora_epoch{RUN_NUMBER}"
print(f"Saving PEFT LoRA adapter weights to {output_path}...")
model.save_pretrained(output_path)
tokenizer.save_pretrained(output_path)
print("Weights saved successfully.")